In [1]:
import os
import sys

# Add the project root to the path
PROJECT_ROOT = '/Users/lekiaprosper/Documents/CoMoChEng/Prometheus/kineticmodelssite'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load environment variables from .env.dev
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env.dev'))

# Configure Django settings
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'kms.settings')

# Allow Django ORM in async context (Jupyter notebooks)
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'

import django
django.setup()

# Fix duplicate output caused by colorama (initialized by Django)
try:
    import colorama
    colorama.deinit()
except:
    pass

In [2]:
from database.models.kinetic_model import *
from database.models.reaction_species import *
from database.models.kinetic_data import *
from database.models.thermo_transport import *
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [3]:
kinetic_models = KineticModel.objects.all()
for model in kinetic_models:
    print(f'Model: {model.model_name}, Species count: {model.species.count()}, Reactions count: {model.kinetics.count()}')


Model: Reduced-DRG-GRI-mech, Species count: 19, Reactions count: 15
Model: PCI2013/325-Husson, Species count: 233, Reactions count: 1401
Model: PCI2013/335-Wang, Species count: 1379, Reactions count: 5636
Model: PCI2013/353-Malewicki, Species count: 345, Reactions count: 2159
Model: PCI2013/269-Matsugi, Species count: 141, Reactions count: 563
Model: PCI2013/289-Dagaut, Species count: 322, Reactions count: 7116
Model: PCI2013/361-Malewicki, Species count: 2013, Reactions count: 7778
Model: PCI2013/411-Darcy, Species count: 942, Reactions count: 4062
Model: PCI2013/599-Veloo, Species count: 113, Reactions count: 0
Model: GRI-17-species-mech, Species count: 17, Reactions count: 56
Model: Chernov, Species count: 64, Reactions count: 0
Model: 2-BTP, Species count: 204, Reactions count: 1495
Model: PCI2015/0153-Marshall, Species count: 20, Reactions count: 66
Model: Narayanaswamy, Species count: 173, Reactions count: 1008
Model: H2-3, Species count: 16, Reactions count: 52
Model: MB-Dooley,

In [4]:
harris_model = kinetic_models.get(model_name='Harris-Butane')
harris_model.model_name

'Harris-Butane'

In [5]:
harris_species_dict = {}
for species in harris_model.species.all():
    for structure_ids in species.isomers.values_list("structure", flat=True):
        structures = Structure.objects.filter(id=structure_ids)
        harris_species_dict[structures.first().isomer.formula.formula] = structures.first().smiles
        # print(structures.first().smiles)


print(harris_species_dict)

{'C4H10': 'CCCC', 'C2H4': 'C=C', 'C3H6': 'C=CC', 'CH3': '[CH3]', 'C2H3O': 'C=C[O]', 'C2H3': '[CH]=C', 'C2H5': 'C[CH2]', 'C2H4O': 'C=CO', 'CH3O2': 'CO[O]', 'C2H5O2': 'CCO[O]', 'C2H6O2': 'CCOO', 'C4H9': 'C[CH]CC', 'C4H9O2': 'C[CH]CCOO', 'C4H10O2': 'CCCCOO', 'C4H8': 'CC=CC', 'C4H7': 'C=C[CH]C', 'C4H9O': 'CCCC[O]', 'C3H8O2': 'CCCOO', 'O2': '[O][O]', 'HO2': '[O]O', 'C2H2O': 'C=C=O', 'H2O2': 'OO', 'H': '[H]', 'H2': '[H][H]', 'CH4': 'C', 'HO': '[OH]', 'H2O': 'O', 'O': '[O]', 'C3H5': '[CH2]C=C', 'C4H10O4': 'OOCCCCOO', 'C4H9O4': '[O]OCCCCOO', 'CH4O2': 'COO', 'C4H8O3': 'O=CCCCOO', 'C4H9O3': '[O]CCCCOO', 'C3H7O2': 'CCCO[O]', 'C4H7O2': '[O]CCCC=O', 'C4H6O2': 'O=CCCC=O', 'C3H4': 'C=C=C'}


In [6]:
# Get all models with the same species in the harris model
for formula, smiles in harris_species_dict.items():
    matching_models = KineticModel.objects.filter(species__isomers__formula__formula=formula).distinct()
    print(f"Models matching {formula}: {matching_models}")
    for model in matching_models:
        print(f" - {model.model_name}")
    print("\n")


Models matching C4H10: <QuerySet [<KineticModel: 222 PCI2019/639-Cai>, <KineticModel: 12 AramcoMech_1.3>, <KineticModel: 217 PCI2017/111-Jin>, <KineticModel: 212 PCI2017/052-Li>, <KineticModel: 27 AramcoMech_2.0>, <KineticModel: 201 PCI2013/599-Veloo>, <KineticModel: 22 2-BTP>, <KineticModel: 31 USC_Mech_ii>, <KineticModel: 108 CombFlame2013/1939-Cai>, <KineticModel: 25 MB-Dooley>, <KineticModel: 44 EL24115>, <KineticModel: 43 Biomass>, <KineticModel: 179 CombFlame2013/2291-Somers>, <KineticModel: 149 PCI2017/024-Bohon>, <KineticModel: 102 CombFlame2013/17-Malewicki>, <KineticModel: 199 PCI2013/361-Malewicki>, <KineticModel: 197 PCI2013/335-Wang>, <KineticModel: 145 PCI2015/0325-Nawdiyal>, <KineticModel: 195 PCI2013/297-Herbinet>, <KineticModel: 196 PCI2013/325-Husson>, '...(remaining elements truncated)...']>
 - PCI2019/639-Cai
 - AramcoMech_1.3
 - PCI2017/111-Jin
 - PCI2017/052-Li
 - AramcoMech_2.0
 - PCI2013/599-Veloo
 - 2-BTP
 - USC_Mech_ii
 - CombFlame2013/1939-Cai
 - MB-Dooley
 -

In [7]:
# Find all models with matching species thermochemistry

# First, find species that match your formulas
for formula, smiles in harris_species_dict.items():
    # Find thermo entries for species matching this formula
    matching_thermo = Thermo.objects.filter(
        species__isomers__formula__formula=formula
    ).distinct()
    
    # Find all KineticModels that use this thermochemistry
    models_with_thermo = KineticModel.objects.filter(
        thermo__in=matching_thermo
    ).distinct()
    
    if models_with_thermo.exists():
        print(f"\n{formula:}")
        for model in models_with_thermo:
            print(f"  - {model.model_name}")



C4H10
  - n-Heptane
  - AramcoMech_1.3
  - 2-BTP
  - MB-Dooley
  - AramcoMech_2.0
  - Gasoline_2
  - USC_Mech_ii
  - Gasoline_Surrogate
  - Biomass
  - EL24115
  - Harris-Butane
  - CombFlame2013/17-Malewicki
  - CombFlame2013/1939-Cai
  - CombFlame2013/2680-Vranckx
  - CombFlame2013/487-Schenk
  - CombFlame2017/176-Needham
  - PCI2013/289-Dagaut
  - PCI2013/401-Liu
  - PCI2013/527-Sheen
  - PCI2015/0325-Nawdiyal
  - PCI2017/024-Bohon
  - PCI2017/037-Sakai
  - CombFlame2012/2028-Sarathy
  - CombFlame2013/1541-Zhang
  - CombFlame2013/1609-Veloo
  - CombFlame2013/2291-Somers
  - CombFlame2013/2712-Sarathy
  - CombFlame2014/65-Darcy
  - CombFlame2014/657-Jin
  - CombFlame2014/798-Cai
  - CombFlame2017/111-Atef
  - CombFlame2019/1-Hanfeng
  - PCI2013/225-Somers
  - PCI2013/297-Herbinet
  - PCI2013/325-Husson
  - PCI2013/335-Wang
  - PCI2013/353-Malewicki
  - PCI2013/361-Malewicki
  - PCI2013/411-Darcy
  - PCI2013/599-Veloo
  - PCI2015/0409-Zhang
  - PCI2017/025-Sudholt
  - PCI2017/032-Che

In [8]:
# Find models sharing the EXACT SAME Thermo database entry
# This means they literally use the same thermo record (same polynomial coefficients)

from collections import defaultdict

# Build a mapping: Thermo ID -> list of models using it
thermo_to_models = defaultdict(list)

for model in KineticModel.objects.all():
    for thermo in model.thermo.all():
        thermo_to_models[thermo.pk].append(model.model_name)

# Find shared thermo entries (used by more than one model)
shared_thermo_data = []
for thermo_id, models in thermo_to_models.items():
    if len(models) > 1:
        thermo = Thermo.objects.get(pk=thermo_id)
        formula = thermo.species.isomers.first().formula.formula if thermo.species.isomers.exists() else "Unknown"
        
        # Get structure identifier
        species = thermo.species
        smiles = ""
        if species.isomers.exists():
            isomer = species.isomers.first()
            if hasattr(isomer, 'structure') and isomer.structure and isomer.structure.smiles:
                smiles = isomer.structure.smiles
        
        shared_thermo_data.append({
            'Thermo ID': thermo_id,
            'Formula': formula,
            'Species': thermo.species.formula,
            'SMILES': smiles[:40] + '...' if len(smiles) > 40 else smiles,
            'Shared By': ', '.join(sorted(set(models))),
            'Model Count': len(set(models))
        })

df_shared = pd.DataFrame(shared_thermo_data)
if not df_shared.empty:
    df_shared = df_shared.sort_values('Model Count', ascending=False)
    
print(f"Found {len(df_shared)} thermo entries shared by multiple models")
df_shared

Found 5447 thermo entries shared by multiple models


,Thermo ID,Formula,Species,SMILES,Shared By,Model Count
69,65,Ar,Ar,,"2-BTP, C1_C3_hydrofluorocarbons_NIST, Chernov,...",48
3001,157,C2H4O,C2H4O,,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, C1_C3_h...",46
409,549,C3H4O,C3H4O,,"AramcoMech_1.3, AramcoMech_2.0, CombFlame2012/...",42
1284,114,C2H2,C2H2,,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, CombFla...",42
3006,168,C3H4,C3H4,,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, Chernov...",39
...,...,...,...,...,...,...
4522,398,C2H3NO,C2H3NO,,NitrogenGlarborg2018,1
4521,397,C2H3N,C2H3N,,NitrogenGlarborg2018,1
4520,396,CH3NO,CH3NO,,NitrogenGlarborg2018,1
4519,395,C2H5N,C2H5N,,NitrogenGlarborg2018,1


In [9]:
import pandas as pd

TEMP = 300  # K - use 300 K since some polynomials have min temp of 300 K

def safe_enthalpy(thermo, temp=TEMP):
    """Safely get enthalpy at specified temperature, return None if out of range"""
    try:
        return thermo.enthalpy(temp)
    except ValueError:
        return None

def get_structure_id(thermo):
    """Get SMILES (preferred) or InChI for a thermo entry's species"""
    species = thermo.species
    if species.isomers.exists():
        isomer = species.isomers.first()
        # Try to get SMILES from Structure
        if hasattr(isomer, 'structure') and isomer.structure:
            if isomer.structure.smiles:
                return ('smiles', isomer.structure.smiles)
        # Fallback to InChI from Isomer
        if isomer.inchi:
            return ('inchi', isomer.inchi)
        # Last resort: formula
        return ('formula', isomer.formula.formula if isomer.formula else 'Unknown')
    return ('unknown', 'Unknown')

# Get Harris-Butane thermo data as reference, keyed by SMILES/InChI
harris_thermo = {}
for thermo in harris_model.thermo.all():
    id_type, struct_id = get_structure_id(thermo)
    formula = thermo.species.isomers.first().formula.formula if thermo.species.isomers.exists() else "Unknown"
    harris_thermo[struct_id] = {
        'species_name': thermo.species.formula,
        'formula': formula,
        'id_type': id_type,
        'enthalpy': safe_enthalpy(thermo),
        'thermo_id': thermo.pk
    }

print(f"Harris-Butane has {len(harris_thermo)} unique species by structure")
print(f"Identifier types: {set(v['id_type'] for v in harris_thermo.values())}")

# Build comparison table
comparison_data = []

for model in KineticModel.objects.exclude(model_name='Harris-Butane'):
    for thermo in model.thermo.all():
        id_type, struct_id = get_structure_id(thermo)
        formula = thermo.species.isomers.first().formula.formula if thermo.species.isomers.exists() else "Unknown"
        
        # Only compare if Harris has this exact structure (SMILES/InChI match)
        if struct_id in harris_thermo:
            harris_h = harris_thermo[struct_id]['enthalpy']
            model_h = safe_enthalpy(thermo)
            
            # Calculate discrepancies
            h_diff = (model_h - harris_h) if (model_h is not None and harris_h is not None) else None
            
            comparison_data.append({
                'Model': model.model_name,
                'Formula': formula,
                'Species': thermo.species.formula,
                'Structure ID': struct_id[:50] + '...' if len(struct_id) > 50 else struct_id,
                'ID Type': id_type,
                f'Harris H({TEMP}K)': harris_h,
                f'Model H({TEMP}K)': model_h,
                f'H Diff (J/mol)': h_diff,
            })

# Create DataFrame
df_enthalpy = pd.DataFrame(comparison_data)

# Sort by absolute discrepancy (largest first)
if not df_enthalpy.empty and f'H Diff (J/mol)' in df_enthalpy.columns:
    df_enthalpy['Abs H Diff'] = df_enthalpy[f'H Diff (J/mol)'].abs()
    df_enthalpy = df_enthalpy.sort_values('Abs H Diff', ascending=False).drop(columns='Abs H Diff')

print(f"\nFound {len(df_enthalpy)} matching thermo entries across models")
df_enthalpy

Harris-Butane has 25 unique species by structure
Identifier types: {'inchi'}

Found 1486 matching thermo entries across models


,Model,Formula,Species,Structure ID,ID Type,Harris H(300K),Model H(300K),H Diff (J/mol)
702,PCI2017/047-Rodriguez,C4H8,C4H8,"InChI=1S/C4H8/c1-3-4-2/h3-4H,1-2H3",inchi,-9560.644506,-541474.073843,-531913.429336
701,PCI2017/047-Rodriguez,C4H7,C4H7,"InChI=1S/C4H7/c1-3-4-2/h3-4H,1H2,2H3/u4",inchi,133913.390193,-281730.483711,-415643.873904
384,GRI-mech-3.0,CH3O2,CH3O2,InChI=1S/CH3O2/c1-3-2/h1H3/u2,inchi,9566.785140,-82995.850016,-92562.635156
391,GRI-mech-3.0,CH3O2,CH3O2,InChI=1S/CH3O2/c1-3-2/h1H3/u2,inchi,9566.785140,-82995.850016,-92562.635156
1411,PCI2013/297-Herbinet,C4H10O2,C4H10O2,"InChI=1S/C4H10O2/c1-2-3-4-6-5/h5H,2-4H2,1H3",inchi,-200554.657495,-231234.764478,-30680.106983
...,...,...,...,...,...,...,...,...
1288,CombFlame2014/84-Wang,C4H9,C4H9,"InChI=1S/C4H9/c1-3-4-2/h3H,4H2,1-2H3/u3",inchi,67554.876806,67544.297865,-10.578941
1336,CombFlame2019/1-Hanfeng,C4H9,C4H9,"InChI=1S/C4H9/c1-3-4-2/h3H,4H2,1-2H3/u3",inchi,67554.876806,67544.297865,-10.578941
905,PCI2017/111-Jin,C4H9,C4H9,"InChI=1S/C4H9/c1-3-4-2/h3H,4H2,1-2H3/u3",inchi,67554.876806,67544.297865,-10.578941
1394,CombFlame2017/176-Needham,C4H9,C4H9,"InChI=1S/C4H9/c1-3-4-2/h3H,4H2,1-2H3/u3",inchi,67554.876806,67544.297865,-10.578941


In [10]:
# Harris model kinetics
# harris_kinetics = harris_model.kinetics.all()

harris_kinetics = KineticsComment.objects.filter(kinetic_model=harris_model)[:20]
for reaction in harris_kinetics:
    reaction = reaction.kinetics.reaction
    print(reaction.equation)

C2H3O + O2 <=> HO2 + C2H2O
C2H5 + HO2 <=> H2O2 + C2H4
C4H10 + H <=> H2 + C4H9
C4H9 + O2 <=> HO2 + C4H8
C4H10 + CH3 <=> CH4 + C4H9
C4H10 + HO <=> H2O + C4H9
C4H10 + O <=> HO + C4H9
C4H9 + O2 <=> HO2 + C4H8
C4H9 + O2 <=> HO2 + C4H8
C2H3O + HO2 <=> H2O2 + C2H2O
2C2H3O <=> C2H2O + C2H4O
C2H3O + C4H9 <=> C4H8 + C2H4O
C4H9 + C3H5 <=> C3H6 + C4H8
C2H3O + C3H5 <=> C2H2O + C3H6
C4H10O4 + C4H9 <=> C4H10 + C4H9O4
C2H3O + CH3O2 <=> C2H2O + CH4O2
C2H3O + C2H5O2 <=> C2H2O + C2H6O2
C4H9O2 + C2H3O <=> C2H2O + C4H10O2
2C2H3O <=> C2H2O + C2H4O
C2H3O + C2H5 <=> C2H4 + C2H4O


In [11]:
import pandas as pd
import numpy as np

# Get Harris-Butane kinetics as reference, keyed by reaction equation
harris_kinetics_dict = {}
for kc in KineticsComment.objects.filter(kinetic_model=harris_model).select_related('kinetics__reaction'):
    kinetics = kc.kinetics
    reaction = kinetics.reaction
    equation = reaction.equation
    
    # Get Arrhenius parameters from raw_data
    raw = kinetics.raw_data
    if raw.get('type') == 'arrhenius':
        harris_kinetics_dict[equation] = {
            'kinetics_id': kinetics.pk,
            'A': raw.get('a_si'),  # pre-exponential in SI units
            'n': raw.get('n'),      # temperature exponent
            'Ea': raw.get('e_si'),  # activation energy in SI (J/mol)
            'A_units': raw.get('a_units'),
            'Ea_units': raw.get('e_units'),
            'type': raw.get('type')
        }

print(f"Harris-Butane has {len(harris_kinetics_dict)} Arrhenius-type reactions")

# Build comparison table - find matching reactions in other models
comparison_data = []

for model in KineticModel.objects.exclude(model_name='Harris-Butane'):
    for kc in KineticsComment.objects.filter(kinetic_model=model).select_related('kinetics__reaction'):
        kinetics = kc.kinetics
        reaction = kinetics.reaction
        equation = reaction.equation
        
        # Only compare if Harris has this exact reaction equation
        if equation in harris_kinetics_dict:
            harris_k = harris_kinetics_dict[equation]
            raw = kinetics.raw_data
            
            if raw.get('type') == 'arrhenius':
                model_A = raw.get('a_si')
                model_n = raw.get('n')
                model_Ea = raw.get('e_si')
                
                # Calculate differences
                A_ratio = model_A / harris_k['A'] if harris_k['A'] and model_A else None
                n_diff = model_n - harris_k['n'] if model_n is not None and harris_k['n'] is not None else None
                Ea_diff = model_Ea - harris_k['Ea'] if model_Ea is not None and harris_k['Ea'] is not None else None
                
                comparison_data.append({
                    'Model': model.model_name,
                    'Reaction': equation[:60] + '...' if len(equation) > 60 else equation,
                    'Harris A (SI)': f"{harris_k['A']:.2e}" if harris_k['A'] else None,
                    'Model A (SI)': f"{model_A:.2e}" if model_A else None,
                    'A Ratio': f"{A_ratio:.3f}" if A_ratio else None,
                    'Harris n': harris_k['n'],
                    'Model n': model_n,
                    'n Diff': round(n_diff, 4) if n_diff is not None else None,
                    'Harris Ea (J/mol)': harris_k['Ea'],
                    'Model Ea (J/mol)': model_Ea,
                    'Ea Diff (J/mol)': round(Ea_diff, 1) if Ea_diff is not None else None,
                })

# Create DataFrame
df_kinetics = pd.DataFrame(comparison_data)

# Sort by largest Ea difference
if not df_kinetics.empty and 'Ea Diff (J/mol)' in df_kinetics.columns:
    df_kinetics['Abs Ea Diff'] = df_kinetics['Ea Diff (J/mol)'].abs()
    df_kinetics = df_kinetics.sort_values('Abs Ea Diff', ascending=False, na_position='last').drop(columns='Abs Ea Diff')

print(f"\nFound {len(df_kinetics)} matching Arrhenius reactions across models")
df_kinetics

Harris-Butane has 29 Arrhenius-type reactions

Found 1520 matching Arrhenius reactions across models


,Model,Reaction,Harris A (SI),Model A (SI),A Ratio,Harris n,Model n,n Diff,Harris Ea (J/mol),Model Ea (J/mol),Ea Diff (J/mol)
332,USC_Mech_ii,C4H9 + O2 <=> HO2 + C4H8,1.55e-03,5.10e+04,32826982.492,3.185290,0.00,-3.1853,210187.0,0.000,-210187.0
1417,CombFlame2017/176-Needham,C4H9 + O2 <=> HO2 + C4H8,1.55e-03,2.70e+05,173789907.312,3.185290,0.00,-3.1853,210187.0,0.000,-210187.0
333,USC_Mech_ii,C4H9 + O2 <=> HO2 + C4H8,1.55e-03,1.20e+05,77239958.805,3.185290,0.00,-3.1853,210187.0,0.000,-210187.0
1299,CombFlame2014/84-Wang,C4H9 + O2 <=> HO2 + C4H8,1.55e-03,5.10e+04,32826982.492,3.185290,0.00,-3.1853,210187.0,0.000,-210187.0
331,USC_Mech_ii,C4H9 + O2 <=> HO2 + C4H8,1.55e-03,2.70e+05,173789907.312,3.185290,0.00,-3.1853,210187.0,0.000,-210187.0
...,...,...,...,...,...,...,...,...,...,...,...
856,PCI2017/111-Jin,C4H10 + H <=> H2 + C4H9,4.82e+03,1.33e+00,0.000,1.437120,2.54,1.1029,28106.4,28267.104,160.7
182,2-BTP,C4H10 + H <=> H2 + C4H9,4.82e+03,9.20e-01,0.000,1.437120,2.54,1.1029,28106.4,28267.104,160.7
12,PCI2013/335-Wang,C4H10 + H <=> H2 + C4H9,4.82e+03,1.81e+00,0.000,1.437120,2.54,1.1029,28106.4,28267.104,160.7
446,EL24115,C4H10 + H <=> H2 + C4H9,4.82e+03,1.81e+00,0.000,1.437120,2.54,1.1029,28106.4,28267.104,160.7


In [12]:
# Find models sharing the EXACT SAME Kinetics database entry
# This means they literally use the same kinetics record (same A, n, Ea)

from collections import defaultdict

# Build a mapping: Kinetics ID -> list of models using it
kinetics_to_models = defaultdict(list)

for model in KineticModel.objects.all():
    for kc in KineticsComment.objects.filter(kinetic_model=model):
        kinetics_to_models[kc.kinetics.pk].append(model.model_name)

# Find shared kinetics entries (used by more than one model)
shared_kinetics_data = []
for kinetics_id, models in kinetics_to_models.items():
    if len(models) > 1:
        kinetics = Kinetics.objects.get(pk=kinetics_id)
        reaction = kinetics.reaction
        raw = kinetics.raw_data
        
        # Get Arrhenius params if available
        A = raw.get('a_si') if raw.get('type') == 'arrhenius' else None
        n = raw.get('n') if raw.get('type') == 'arrhenius' else None
        Ea = raw.get('e_si') if raw.get('type') == 'arrhenius' else None
        
        shared_kinetics_data.append({
            'Kinetics ID': kinetics_id,
            'Reaction': reaction.equation[:50] + '...' if len(reaction.equation) > 50 else reaction.equation,
            'Type': raw.get('type'),
            'A (SI)': f"{A:.2e}" if A else None,
            'n': n,
            'Ea (J/mol)': Ea,
            'Shared By': ', '.join(sorted(set(models))),
            'Model Count': len(set(models))
        })

df_shared_kinetics = pd.DataFrame(shared_kinetics_data)
if not df_shared_kinetics.empty:
    df_shared_kinetics = df_shared_kinetics.sort_values('Model Count', ascending=False)

print(f"Found {len(df_shared_kinetics)} kinetics entries shared by multiple models")
df_shared_kinetics

Found 26825 kinetics entries shared by multiple models


,Kinetics ID,Reaction,Type,A (SI),n,Ea (J/mol),Shared By,Model Count
143,46,CHO + O <=> H + CO2,arrhenius,3.00e+07,0.0,0.0,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, Biomass...",70
103,1651,C + HO <=> CO + H,arrhenius,5.00e+07,0.0,0.0,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, Biomass...",63
104,33,CH + HO <=> H + CHO,arrhenius,3.00e+07,0.0,0.0,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, Biomass...",62
1494,818,CH + CH4 <=> H + C2H4,arrhenius,6.00e+07,0.0,0.0,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, Biomass...",61
1323,54,CH3O + H <=> H2 + CH2O,arrhenius,2.00e+07,0.0,0.0,"2-BTP, AramcoMech_1.3, AramcoMech_2.0, Biomass...",59
...,...,...,...,...,...,...,...,...
15858,23946,C11H23O2 + C11H24 <=> C11H23 + C11H24O2,arrhenius,1.00e+05,0.0,43513.6,"CombFlame2013/17-Malewicki, PCI2013/361-Malewicki",2
15859,23947,C11H23O2 + C11H24 <=> C11H23 + C11H24O2,arrhenius,1.00e+05,0.0,43513.6,"CombFlame2013/17-Malewicki, PCI2013/361-Malewicki",2
15860,23948,C11H23O2 + C11H24 <=> C11H23 + C11H24O2,arrhenius,5.00e+04,0.0,43513.6,"CombFlame2013/17-Malewicki, PCI2013/361-Malewicki",2
15861,23949,C11H23O2 + C11H24 <=> C11H23 + C11H24O2,arrhenius,1.00e+05,0.0,56065.6,"CombFlame2013/17-Malewicki, PCI2013/361-Malewicki",2


In [ ]:
# Find all models that share the same Reaction objects as Harris-Butane
# and compare their kinetics (rate constants)

import pandas as pd
import numpy as np

R = 8.314  # J/(mol·K)
TEMP_COMPARE = 1000  # K - typical combustion temperature for rate constant comparison

def rate_constant(A, n, Ea, T):
    """Calculate Arrhenius rate constant k = A * T^n * exp(-Ea/(RT))"""
    if A is None or n is None or Ea is None:
        return None
    try:
        return A * (T ** n) * np.exp(-Ea / (R * T))
    except (OverflowError, FloatingPointError):
        return None

# Step 1: Get all Harris reaction IDs and their kinetics
harris_reactions = {}  # reaction_id -> {kinetics info}
for kc in KineticsComment.objects.filter(kinetic_model=harris_model).select_related('kinetics__reaction'):
    kinetics = kc.kinetics
    reaction = kinetics.reaction
    raw = kinetics.raw_data
    
    harris_reactions[reaction.pk] = {
        'equation': reaction.equation,
        'kinetics_id': kinetics.pk,
        'type': raw.get('type'),
        'A': raw.get('a_si'),
        'n': raw.get('n'),
        'Ea': raw.get('e_si'),
    }

print(f"Harris-Butane has {len(harris_reactions)} reactions")

# Step 2: Find all other models that use the SAME Reaction objects
comparison_data = []

for model in KineticModel.objects.exclude(model_name='Harris-Butane'):
    for kc in KineticsComment.objects.filter(kinetic_model=model).select_related('kinetics__reaction'):
        kinetics = kc.kinetics
        reaction = kinetics.reaction
        raw = kinetics.raw_data
        
        # Only compare if Harris has this exact same Reaction object
        if reaction.pk in harris_reactions:
            harris_k = harris_reactions[reaction.pk]
            
            model_A = raw.get('a_si')
            model_n = raw.get('n')
            model_Ea = raw.get('e_si')
            
            # Calculate rate constants at comparison temperature
            harris_rate = rate_constant(harris_k['A'], harris_k['n'], harris_k['Ea'], TEMP_COMPARE)
            model_rate = rate_constant(model_A, model_n, model_Ea, TEMP_COMPARE)
            
            # Rate constant ratio
            k_ratio = model_rate / harris_rate if (harris_rate and model_rate and harris_rate != 0) else None
            log_k_ratio = np.log10(abs(k_ratio)) if k_ratio and k_ratio > 0 else None
            
            # Parameter differences
            A_ratio = model_A / harris_k['A'] if (harris_k['A'] and model_A and harris_k['A'] != 0) else None
            n_diff = (model_n - harris_k['n']) if (model_n is not None and harris_k['n'] is not None) else None
            Ea_diff = (model_Ea - harris_k['Ea']) if (model_Ea is not None and harris_k['Ea'] is not None) else None
            
            comparison_data.append({
                'Model': model.model_name,
                'Reaction': reaction.equation[:70] + '...' if len(reaction.equation) > 70 else reaction.equation,
                'Reaction ID': reaction.pk,
                'Type (Harris)': harris_k['type'],
                'Type (Model)': raw.get('type'),
                f'Harris k({TEMP_COMPARE}K)': f"{harris_rate:.3e}" if harris_rate else None,
                f'Model k({TEMP_COMPARE}K)': f"{model_rate:.3e}" if model_rate else None,
                f'k Ratio': f"{k_ratio:.4f}" if k_ratio else None,
                f'log10(k Ratio)': round(log_k_ratio, 3) if log_k_ratio is not None else None,
                'A Ratio': f"{A_ratio:.4f}" if A_ratio else None,
                'n Diff': round(n_diff, 4) if n_diff is not None else None,
                'Ea Diff (kJ/mol)': round(Ea_diff / 1000, 2) if Ea_diff is not None else None,
            })

# Create DataFrame
df_kinetics_compare = pd.DataFrame(comparison_data)

# Sort by largest rate constant deviation
if not df_kinetics_compare.empty and 'log10(k Ratio)' in df_kinetics_compare.columns:
    df_kinetics_compare['Abs log k'] = df_kinetics_compare['log10(k Ratio)'].abs()
    df_kinetics_compare = df_kinetics_compare.sort_values('Abs log k', ascending=False, na_position='last').drop(columns='Abs log k')

print(f"\nFound {len(df_kinetics_compare)} matching reactions across models")

# Summary: how many matching reactions per model
if not df_kinetics_compare.empty:
    summary = df_kinetics_compare.groupby('Model').agg(
        Matching_Reactions=('Reaction', 'count'),
        Mean_Abs_log_k=('log10(k Ratio)', lambda x: x.abs().mean()),
        Max_Abs_log_k=('log10(k Ratio)', lambda x: x.abs().max()),
    ).sort_values('Matching_Reactions', ascending=False)
    print(f"\n--- Summary by Model ---")
    display(summary)

df_kinetics_compare

Harris-Butane has 42 reactions

Found 440 matching reactions across models
Rate constants compared at 1000 K

--- Summary by Model ---


,Matching_Reactions,Mean_Abs_log_k,Max_Abs_log_k
Model,,,
CombFlame2013/2680-Vranckx,24,2.262542,7.053
PCI2013/289-Dagaut,11,2.807364,9.577
CombFlame2013/1609-Veloo,11,3.510273,9.599
PCI2017/051-Pelucchi,11,2.807364,9.577
CombFlame2013/487-Schenk,11,3.961636,9.846
...,...,...,...
CombFlame2013/1907-Merchant,1,4.974000,4.974
CombFlame2013/2343-Hansen,1,4.974000,4.974
CombFlame2014/711-Allen,1,4.974000,4.974


,Model,Reaction,Reaction ID,Type (Harris),Type (Model),Harris k(1000K),Model k(1000K),k Ratio,log10(k Ratio),A Ratio,n Diff,Ea Diff (kJ/mol)
17,PCI2013/353-Malewicki,C4H9 + O2 <=> HO2 + C4H8,12058,arrhenius,arrhenius,3.356e-04,1.615e-25,0.0000,-21.318,0.0000,-3.031,-176.38
64,MB-Dooley,C4H9 + O2 <=> HO2 + C4H8,12058,arrhenius,arrhenius,3.356e-04,1.615e-25,0.0000,-21.318,0.0000,-3.031,-176.38
130,CombFlame2013/17-Malewicki,C4H9 + O2 <=> HO2 + C4H8,12058,arrhenius,arrhenius,3.356e-04,1.615e-25,0.0000,-21.318,0.0000,-3.031,-176.38
37,PCI2013/361-Malewicki,C4H9 + O2 <=> HO2 + C4H8,12058,arrhenius,arrhenius,3.356e-04,1.615e-25,0.0000,-21.318,0.0000,-3.031,-176.38
103,Gasoline_Surrogate,C4H9 + O2 <=> HO2 + C4H8,12058,arrhenius,arrhenius,3.356e-04,1.615e-25,0.0000,-21.318,0.0000,-3.031,-176.38
...,...,...,...,...,...,...,...,...,...,...,...,...
189,PCI2017/047-Rodriguez,C4H10 + HO <=> H2O + C4H9,12019,arrhenius,multi_arrhenius,1.348e+07,None,None,NaN,None,NaN,NaN
190,PCI2017/047-Rodriguez,C4H10 + CH3 <=> CH4 + C4H9,12015,arrhenius,multi_arrhenius,7.111e+04,None,None,NaN,None,NaN,NaN
419,PCI2013/297-Herbinet,C4H9 + O2 <=> HO2 + C4H8,12057,arrhenius,multi_arrhenius,5.858e-05,None,None,NaN,None,NaN,NaN
420,PCI2013/297-Herbinet,C4H10 + HO <=> H2O + C4H9,12019,arrhenius,multi_arrhenius,1.348e+07,None,None,NaN,None,NaN,NaN
